# Validation And Performance Gauntlet

This notebook answers three practical questions about the current MD/MM engine:

1. Do the implemented force terms agree with finite-difference forces?
2. Do short NVE/NVT trajectories stay numerically sane?
3. Which current paths look expensive enough to deserve lower-level optimization?

The goal is not to produce publication-quality physics from tiny toy systems. The goal is to create repeatable engineering evidence before we start writing custom Metal kernels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.benchmarks import mm_force_terms, stability
from mlx_atomistic.runtime import get_runtime_info
from mlx_atomistic.validation import run_force_validation_suite, summarize_validation_results

runtime = get_runtime_info()
runtime

## 1. Force correctness

Forces are what MD actually integrates. A potential can return a plausible energy while still returning the wrong force sign, wrong pair scaling, or unstable gradient near a geometric singularity. The finite-difference check perturbs every coordinate and compares `F = -∂E/∂x` against the force returned by each force term.

In [ ]:
validation_results = run_force_validation_suite(seed=7, cases_per_term=1)
validation_summary = summarize_validation_results(validation_results)
validation_summary

In [ ]:
validation_rows = [result.to_dict() for result in validation_results]
labels = [row["case_name"] for row in validation_rows]
errors = [row["max_abs_error"] for row in validation_rows]
tolerances = [row["tolerance"] for row in validation_rows]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(labels, errors, label="max |ΔF|")
ax.plot(labels, tolerances, color="black", marker="o", linewidth=1.5, label="tolerance")
ax.set_yscale("log")
ax.set_ylabel("force error")
ax.set_title("Finite-difference force validation")
ax.legend()
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

Read this plot as a correctness gate. Bars below the tolerance line are acceptable for this finite-difference step size. If a refactor pushes a bar above the line, we should debug the force expression before trusting trajectory behavior.

## 2. Short stability runs

NVE should approximately conserve total energy when `dt` is small enough. NVT does not conserve total energy because the thermostat exchanges energy with the system, so for NVT we watch temperature, reproducibility, and nonfinite values instead.

In [ ]:
stability_payload = stability.build_payload(
    sizes=[32],
    steps=20,
    bonded_steps=50,
    dt_values=[0.001, 0.002, 0.004],
    temperature=1.0,
    seed=7,
)
stability_payload["summary"]

In [ ]:
stability_rows = stability_payload["cases"]
case_labels = [f"{row['case']}\n{row['ensemble']} dt={row['dt']}" for row in stability_rows]
relative_drift = [row["relative_energy_drift"] for row in stability_rows]
mean_temperature = [row["mean_temperature"] for row in stability_rows]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(case_labels, relative_drift, color="#386cb0")
axes[0].set_yscale("log")
axes[0].set_ylabel("relative energy drift")
axes[0].set_title("NVE/NVT energy movement")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(case_labels, mean_temperature, color="#fdb462")
axes[1].axhline(1.0, color="black", linewidth=1.2, linestyle="--", label="target T")
axes[1].set_ylabel("mean reduced temperature")
axes[1].set_title("Temperature behavior")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()
fig.tight_layout()

The bonded-chain rows show how the integrator responds as `dt` changes. The LJ rows exercise the neighbor-list path and give us a quick signal for nonfinite energies, unstable temperatures, or rebuild behavior.

## 3. Hot-path candidates

This benchmark separates the obvious optimization candidates: bonded autodiff, neighbor-list construction, LJ pair-list evaluation, and direct cutoff Coulomb. These are not final performance numbers; they are a guide for choosing the next implementation target.

In [ ]:
profile_results = mm_force_terms.run_benchmark(evaluations=3, particles=64)
profile_rows = [result.to_dict() for result in profile_results]
profile_rows

In [ ]:
labels = [f"{row['category']}\n{row['term']}" for row in profile_rows]
timings = [row["ms_per_eval"] for row in profile_rows]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(labels, timings, color="#7fc97f")
ax.set_ylabel("ms / evaluation")
ax.set_title("Force-path timing breakdown")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()

If one bar dominates consistently at realistic sizes, that is where the first custom kernel or algorithmic refactor belongs. Until then, the best optimization work is better measurement.